In [2]:
# Cell 1 — Setup & imports
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

import matplotlib.pyplot as plt

# xgboost
import xgboost as xgb
from xgboost import XGBClassifier

# local helpers
import sys
sys.path.append("..")
from src.transforms import (
    get_preprocessor_fixed,
    compute_categories_list,
    select_cols_indices,
    get_feature_names_from_preprocessor
)
from src.artifacts import save_joblib_with_manifest, save_json_with_manifest, ensure_artifact_dirs

# constants / paths (adjust if your layout differs)
SEED = 42
K = 15  # chosen k (we decided to go with k=15)
ART_ROOT = "../artifacts"
FS_DIR = os.path.join(ART_ROOT, "feature_selection")
TUNE_DIR = os.path.join(ART_ROOT, "tuning_optuna")
PREPROC_META_DIR = os.path.join(ART_ROOT, "preprocessor")
OUT_DIR = os.path.join(TUNE_DIR, "calibration")
os.makedirs(OUT_DIR, exist_ok=True)
ensure_artifact_dirs()

print("Paths:")
print(" feature_index_map:", os.path.join(FS_DIR, "feature_index_map.json"))
print(" tuning dir:", TUNE_DIR)
print(" preproc meta:", PREPROC_META_DIR)
print(" calibration out:", OUT_DIR)


Paths:
 feature_index_map: ../artifacts\feature_selection\feature_index_map.json
 tuning dir: ../artifacts\tuning_optuna
 preproc meta: ../artifacts\preprocessor
 calibration out: ../artifacts\tuning_optuna\calibration


In [4]:
# Cell 2 — load data & meta
train_csv = os.path.join(ART_ROOT, "data", "train.csv")
hold_csv  = os.path.join(ART_ROOT, "data", "holdout.csv")
featmap_path = os.path.join(FS_DIR, "feature_index_map.json")
feature_ranks_csv = os.path.join(FS_DIR, "feature_ranks.csv")

assert Path(train_csv).exists(), f"Missing train.csv at {train_csv}"
assert Path(hold_csv).exists(), f"Missing holdout.csv at {hold_csv}"
assert Path(featmap_path).exists(), f"Missing feature_index_map.json at {featmap_path}"

train_df = pd.read_csv(train_csv)
hold_df  = pd.read_csv(hold_csv)

X_train_full = train_df.drop(columns=["num"])
y_train_full = train_df["num"].values

X_hold = hold_df.drop(columns=["num"])
y_hold = hold_df["num"].values

featmap = json.load(open(featmap_path, "r"))
full_feature_names = featmap["full_feature_names"]
categorical_features = featmap.get("categorical_features", None)
numeric_features = featmap.get("numeric_features", None)
categories_list = featmap.get("categories_list", None)

# If categories_list not present in featmap, try reading preprocessor categories
if categories_list is None:
    cat_meta_path = os.path.join(PREPROC_META_DIR, "categories_list.json")
    if Path(cat_meta_path).exists():
        meta = json.load(open(cat_meta_path, "r"))
        categories_list = meta.get("categories_list", None)

print("Data shapes:", X_train_full.shape, X_hold.shape)
print("Full transformed feature count:", len(full_feature_names))


Data shapes: (736, 15) (184, 15)
Full transformed feature count: 25


In [6]:
# Cell 3 — compute top-K features (from feature_ranks)
if Path(feature_ranks_csv).exists():
    fr = pd.read_csv(feature_ranks_csv)
    top_features = fr.sort_values("rank")["feature"].tolist()[:K]
else:
    # fallback: take first K features of full_feature_names
    top_features = full_feature_names[:K]

top_indices = [full_feature_names.index(f) for f in top_features]
print(f"Top-{K} feature names (selected):\n", top_features)
print("Indices in transformed array:", top_indices)


Top-15 feature names (selected):
 ['cp_asymptomatic', 'cp_atypical angina', 'sex_Female', 'thal_normal', 'ca', 'chol', 'oldpeak', 'age', 'slope_flat', 'thal_reversable defect', 'restecg_normal', 'cp_typical angina', 'cp_non-anginal', 'thalch', 'restecg_st-t abnormality']
Indices in transformed array: [8, 9, 6, 23, 5, 2, 4, 0, 20, 24, 15, 11, 10, 3, 16]


In [8]:
# Cell 4 — build preprocessor_fixed using categories_list (must exist)
if categories_list is None:
    # compute categories_list from training data (safe fallback)
    print("categories_list not found in featmap; computing from training data (deterministic).")
    categories_list = compute_categories_list(X_train_full, featmap.get("categorical_features", None))

preproc_fixed = get_preprocessor_fixed(categories_list=categories_list)
print("Preprocessor (unfitted) created. Fitting on training subset later...")

# portable selector for transformed columns (select_cols_indices in src.transforms)
selector = FunctionTransformer(func=select_cols_indices, kw_args={"indices": top_indices}, validate=False)


Preprocessor (unfitted) created. Fitting on training subset later...


In [9]:
# Cell 5 — create calibration split from full training set
# We'll use train_sub for training base estimator and calib set for calibrator (prefit mode)
train_sub_frac = 0.80  # 80% for base training, 20% for calibration
X_train_sub, X_calib, y_train_sub, y_calib = train_test_split(
    X_train_full, y_train_full, test_size=1-train_sub_frac, random_state=SEED, stratify=y_train_full
)

print("train_sub:", X_train_sub.shape, "calib:", X_calib.shape)


train_sub: (588, 15) calib: (148, 15)


In [10]:
# Cell 6 — prepare base pipeline and load tuned XGB params/model if available

# 1) attempt to load a saved tuned pipeline or best params from tuning directory
best_pipeline_path = os.path.join(TUNE_DIR, f"xgb_k{K}_pipeline.joblib")
best_booster_bst = os.path.join(TUNE_DIR, f"xgb_k{K}_final.bst")
best_pipeline_joblib = os.path.join(TUNE_DIR, f"xgb_k{K}_final.joblib")
best_params_json = os.path.join(TUNE_DIR, f"xgb_k{K}_best_params.json")

xgb_clf = None

# If a pipeline joblib exists, load it and use (it contains preprocessor inside if saved that way)
if Path(best_pipeline_joblib).exists():
    print("Loading full saved pipeline:", best_pipeline_joblib)
    pipeline_loaded = joblib.load(best_pipeline_joblib)
    # We'll use this as the base estimator for calibration
    base_pipeline = pipeline_loaded
else:
    # Otherwise construct pipeline with preprocessor_fixed and XGBClassifier.
    # Try to get tuned params from JSON
    params = {}
    if Path(best_params_json).exists():
        try:
            params = json.load(open(best_params_json, "r"))
        except Exception:
            params = {}
    # fallback reasonable defaults:
    xgb_params = dict(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=SEED,
        verbosity=0
    )
    xgb_params.update(params)

    # instantiate classifier
    xgb_clf = XGBClassifier(**xgb_params)
    # if a .bst exists, try to load into classifier
    if Path(best_booster_bst).exists():
        try:
            print("Found .bst booster file; loading into XGBClassifier using .load_model()")
            xgb_clf.load_model(best_booster_bst)
            # best_iteration may exist in metadata, but loaded model will work for predict_proba
        except Exception as ex:
            print("Warning: couldn't load .bst into XGBClassifier:", ex)

    base_pipeline = Pipeline([
        ("preproc", preproc_fixed),
        ("selector", selector),
        ("xgb", xgb_clf)
    ])

print("Base pipeline ready. Note: estimator inside may be unfitted (we'll fit below if needed).")


Found .bst booster file; loading into XGBClassifier using .load_model()
Base pipeline ready. Note: estimator inside may be unfitted (we'll fit below if needed).


C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\xgboost\sklearn.py:1125: UserWarning: [08:51:17] WARNING: D:\bld\xgboost-split_1764148481205\work\src\c_api\c_api.cc:1511: Unknown file format: `bst`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)


In [15]:
# Cell 7 — fit base pipeline if necessary
# If we loaded a full saved pipeline (prefit), it is already fitted. Otherwise, fit the pipeline now.

def pipeline_is_fitted(pipe):
    try:
        # check if final estimator has fitted attributes
        last = pipe.named_steps[list(pipe.named_steps.keys())[-1]]
        # many sklearn estimators have 'classes_' after fit
        return hasattr(last, "classes_") or hasattr(last, "booster_") or hasattr(last, "n_features_in_")
    except Exception:
        return False

if not pipeline_is_fitted(base_pipeline):
    print("Fitting base pipeline on train_sub (no leakage). This fits preprocessor + xgb.")
    base_pipeline.fit(X_train_sub, y_train_sub)
    # save pipeline for reproducibility
    save_path = os.path.join(OUT_DIR, f"base_pipeline_k{K}.joblib")
    joblib.dump(base_pipeline, save_path)
    print("Saved base pipeline to:", save_path)
else:
    print("Base pipeline already fitted (loaded).")


Base pipeline already fitted (loaded).


In [16]:
# Cell 8 — Calibrate (Platt / sigmoid). Use cv='prefit' and the separate calibration set.
print("Calibrating using cv='prefit' and method='sigmoid' (Platt scaling).")
calibrator = CalibratedClassifierCV(estimator=base_pipeline, method="sigmoid", cv="prefit")
calibrator.fit(X_calib, y_calib)

# Save calibrated model
calib_path = os.path.join(OUT_DIR, f"xgb_k{K}_calibrated_sigmoid.joblib")
joblib.dump(calibrator, calib_path)
save_json_with_manifest({"k": K, "calibrated_method": "sigmoid", "path": calib_path}, os.path.join(OUT_DIR, f"xgb_k{K}_calibrated_meta.json"), name=f"xgb_k{K}_calibrated_meta")
print("Saved calibrated model:", calib_path)


Calibrating using cv='prefit' and method='sigmoid' (Platt scaling).


C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


NotFittedError: Pipeline is not fitted yet.